In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, round

In [0]:
pay_df = spark.table("01_global_tech_bronze.raw_tables.payroll_raw")

transformed_pay_df = (
    pay_df
    .withColumn("gross_salary", col("gross_salary").cast("double"))
    .withColumn("bonus", col("bonus").cast("double"))
    .withColumn("overtime_pay", col("overtime_pay").cast("double"))
    .withColumn("commission", col("commission").cast("double"))
    .withColumn("allowances", col("allowances").cast("double"))
    .withColumn(
        "total_compensation",
        (col("gross_salary") + col("bonus") + col("overtime_pay") + col("commission") + col("allowances")).cast("double")
    )
    .withColumn("total_compensation", round(col("total_compensation"), 2))
    .withColumn("payroll_sk", monotonically_increasing_id())
    .withColumnRenamed("status", "payroll_status")
)

transformed_pay_df.write.format("delta").mode("overwrite").saveAsTable("02_global_tech_silver.transformed_tables.transformed_payroll")

display(transformed_pay_df)